# 🎯 DipRadar — ML Training (Colab)

Pipeline completo de treino dos modelos ML que correm no Railway.

**Outputs gerados e pushed para GitHub:**
- `dip_models_v3.pkl` → bundle do modelo (joblib)
- `ml_report_v3.json` → métricas e metadados do treino
- `ml_training_base.parquet` → dataset de treino completo

**Fluxo (correr em ordem):**
1. Célula 1 — Secrets + clone do repo + install deps
2. Célula 2 — Download de preços via Tiingo (ETFs + tickers do universo)
3. Célula 3 — Build do dataset de treino (features + targets)
4. Célula 4 — Walk-forward CV + seleção do champion
5. Célula 5 — Treino full + calibrador isotónico
6. Célula 6 — Bundle + Report
7. Célula 7 — Guardar parquet actualizado
8. Célula 8 — Push automático para GitHub (Railway redeploy automático)

## ⚙️ Célula 1 — Setup: secrets, clone, install deps

In [ ]:
# ── Secrets do Colab (Google Colab → ícone 🔑 Secrets)
# Necessário definir: GITHUB_TOKEN, TIINGO_API_KEY
import os, subprocess, sys
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
TIINGO_KEY   = userdata.get('TIINGO_API_KEY')
REPO_OWNER   = 'romeurf'
REPO_NAME    = 'DipRadar'
BRANCH       = 'main'

assert GITHUB_TOKEN, 'Falta GITHUB_TOKEN nos secrets do Colab'
assert TIINGO_KEY,   'Falta TIINGO_API_KEY nos secrets do Colab'
print('✅ Secrets carregados')

# Clone autenticado
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth=1', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'✅ Repo em {REPO_DIR}')

# Install dependências
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'joblib', 'scikit-learn', 'lightgbm', 'xgboost',
                'pandas', 'pyarrow', 'requests', 'imbalanced-learn'], check=True)
print('✅ Dependências instaladas')

## 🌐 Célula 2 — Download de preços via Tiingo

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime

os.environ['TIINGO_API_KEY'] = TIINGO_KEY
REPO_PATH = Path(REPO_DIR)

def tiingo_daily(ticker: str, start: str = '2018-01-01') -> pd.DataFrame:
    """Fetch OHLCV diário do Tiingo. Devolve DataFrame indexado por Date."""
    url = f'https://api.tiingo.com/tiingo/daily/{ticker}/prices'
    params = {
        'startDate': start,
        'endDate': datetime.today().strftime('%Y-%m-%d'),
        'token': TIINGO_KEY,
        'resampleFreq': 'daily',
        'columns': 'date,open,high,low,close,volume,adjClose'
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low',
        'close': 'Close', 'volume': 'Volume', 'adjClose': 'AdjClose'
    })
    return df.sort_index()

# ETFs de referência
from ml_training.config import DEFAULT_ETF, SECTOR_ETF

etf_tickers = list(set([DEFAULT_ETF] + list(SECTOR_ETF.values())))
print(f'ETFs a descarregar: {sorted(etf_tickers)}')

etf_cache: dict[str, pd.DataFrame] = {}
for etf in etf_tickers:
    try:
        df_etf = tiingo_daily(etf)
        if not df_etf.empty:
            etf_cache[etf] = df_etf
            print(f'  ✅ {etf}: {len(df_etf)} candles ({df_etf.index.min().date()} → {df_etf.index.max().date()})')
        else:
            print(f'  ⚠️  {etf}: sem dados')
    except Exception as e:
        print(f'  ❌ {etf}: {e}')

print(f'\n✅ ETF cache: {len(etf_cache)}/{len(etf_tickers)} ETFs carregados')

In [ ]:
# Tickers do universo DipRadar — obtidos do universe.py ou fallback manual
# Tenta carregar do módulo universe; se falhar usa lista manual
try:
    from universe import get_universe_tickers
    all_tickers = sorted(get_universe_tickers())
    print(f'Tickers do universe.py: {len(all_tickers)}')
except Exception as e:
    print(f'universe.py não disponível ({e}) — a usar lista manual')
    all_tickers = sorted([
        'AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA','BRK-B',
        'JNJ','UNH','LLY','ABBV','MRK','ABT','TMO','DHR',
        'JPM','BAC','WFC','GS','MS','BLK',
        'PG','KO','PEP','WMT','COST','MCD','HD','TGT',
        'XOM','CVX','COP','SLB',
        'NEE','DUK','SO',
        'CAT','DE','HON','MMM','GE','RTX',
        'VZ','T','CMCSA',
        'AMT','PLD','EQIX',
        'LIN','APD','SHW',
        'SPY','QQQ','IWM'
    ])
    print(f'Tickers manuais: {len(all_tickers)}')

print(f'\nA descarregar preços de {len(all_tickers)} tickers...')
price_cache: dict[str, pd.DataFrame] = {}
errors = []

for i, tk in enumerate(all_tickers):
    if (i + 1) % 25 == 0:
        print(f'  ... {i+1}/{len(all_tickers)}')
    try:
        df_p = tiingo_daily(tk)
        if not df_p.empty:
            price_cache[tk] = df_p
    except Exception as e:
        errors.append((tk, str(e)))

print(f'\n✅ Price cache: {len(price_cache)}/{len(all_tickers)} tickers')
if errors:
    print(f'⚠️  Erros ({len(errors)}): {errors[:5]}')

## 🏗️ Célula 3 — Build do dataset de treino (features + targets)

In [ ]:
# Build do dataset de treino directamente a partir dos preços
# Não depende de nenhum parquet pré-existente nem de experiments/
import numpy as np
import math

from ml_training.config import (
    DEFAULT_ETF, SECTOR_ETF, HORIZON_DAYS,
    MOMENTUM_FEATURES, NEW_FEATURES_V31
)
from ml_features import (
    FEATURE_COLUMNS, MOMENTUM_FEATURE_COLUMNS,
    _FALLBACK, add_derived_features, add_momentum_features
)

# Definição local de build_v2_features e build_targets
# (elimina dependência de experiments/ml_v2/pipeline.py que não existe)

def _rsi(closes: pd.Series, period: int = 14) -> float:
    """RSI simples sobre uma série de closes."""
    if len(closes) < period + 1:
        return float(_FALLBACK.get('rsi_14', 50.0))
    delta = closes.diff().dropna()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    rsi_val = (100 - 100 / (1 + rs)).iloc[-1]
    return float(rsi_val) if pd.notna(rsi_val) else float(_FALLBACK.get('rsi_14', 50.0))

def _atr_ratio(hist: pd.DataFrame, period: int = 14) -> float:
    """ATR(14) / Close — volatilidade normalizada."""
    if len(hist) < period + 1:
        return float(_FALLBACK.get('atr_ratio', 0.02))
    tr = pd.concat([
        hist['High'] - hist['Low'],
        (hist['High'] - hist['Close'].shift()).abs(),
        (hist['Low'] - hist['Close'].shift()).abs()
    ], axis=1).max(axis=1)
    atr = tr.rolling(period).mean().iloc[-1]
    close = hist['Close'].iloc[-1]
    return float(atr / close) if close > 0 else float(_FALLBACK.get('atr_ratio', 0.02))

def _volume_spike(hist: pd.DataFrame, window: int = 20) -> float:
    """Volume do último dia / média 20d."""
    if len(hist) < window:
        return 1.0
    avg = hist['Volume'].iloc[-window:].mean()
    last = hist['Volume'].iloc[-1]
    return float(last / avg) if avg > 0 else 1.0

def build_v2_features(hist: pd.DataFrame, alert_date: pd.Timestamp) -> dict:
    """Calcula features price-based v2 directamente do OHLCV."""
    h = hist[hist.index <= alert_date]
    if h.empty:
        return {}
    close = h['Close'].iloc[-1]
    high_52w = h['High'].iloc[-252:].max() if len(h) >= 5 else close
    low_52w = h['Low'].iloc[-252:].min() if len(h) >= 5 else close
    prev_close = h['Close'].iloc[-2] if len(h) >= 2 else close
    drop_today = (close / prev_close - 1) if prev_close > 0 else 0.0
    drawdown_52w = (close / high_52w - 1) if high_52w > 0 else 0.0
    return {
        'drop_pct_today': float(drop_today),
        'drawdown_52w': float(drawdown_52w),
        'rsi_14': _rsi(h['Close']),
        'atr_ratio': _atr_ratio(h),
        'volume_spike': _volume_spike(h),
    }

def build_targets(alert_date: pd.Timestamp, hist: pd.DataFrame, horizon: int = HORIZON_DAYS) -> dict:
    """Calcula max_return_60d e max_drawdown_60d forward-only."""
    entry_slice = hist[hist.index <= alert_date]
    if entry_slice.empty:
        return {'max_return_60d': float('nan'), 'max_drawdown_60d': float('nan')}
    entry_price = float(entry_slice['Close'].iloc[-1])
    if entry_price <= 0:
        return {'max_return_60d': float('nan'), 'max_drawdown_60d': float('nan')}
    fwd = hist[
        (hist.index > alert_date) &
        (hist.index <= alert_date + pd.Timedelta(days=horizon))
    ]
    if len(fwd) < 5:
        return {'max_return_60d': float('nan'), 'max_drawdown_60d': float('nan')}
    max_ret = float(fwd['High'].max() / entry_price - 1)
    max_draw = float(fwd['Low'].min() / entry_price - 1)
    return {'max_return_60d': max_ret, 'max_drawdown_60d': max_draw}

def spy_max_return_forward(spy_hist, alert_date, horizon=HORIZON_DAYS):
    """SPY max return forward — para calcular alpha_60d."""
    if spy_hist is None:
        return float('nan')
    entry_slice = spy_hist[spy_hist.index <= alert_date]
    if entry_slice.empty:
        return float('nan')
    spy_entry = float(entry_slice['Close'].iloc[-1])
    if spy_entry <= 0:
        return float('nan')
    fwd = spy_hist[
        (spy_hist.index > alert_date) &
        (spy_hist.index <= alert_date + pd.Timedelta(days=horizon))
    ]
    if len(fwd) < 5:
        return float('nan')
    return float(fwd['Close'].max() / spy_entry - 1)

def days_since_52w_high(hist: pd.DataFrame, alert_date: pd.Timestamp) -> float:
    window = hist[
        (hist.index <= alert_date) &
        (hist.index > alert_date - pd.Timedelta(days=365))
    ]
    if len(window) < 20:
        return 60.0
    high_idx = window['High'].idxmax()
    return float((alert_date - high_idx).days)

print('✅ Funções auxiliares definidas')

# Lista de features totais
NEW_FEATURES = ['relative_drop', 'sector_alert_count_7d', 'days_since_52w_high', 'month_of_year']
FEATURE_COLS_V31 = FEATURE_COLUMNS + MOMENTUM_FEATURE_COLUMNS + NEW_FEATURES

print(f'Features totais: {len(FEATURE_COLS_V31)}')
print(f'  Base:     {len(FEATURE_COLUMNS)}')
print(f'  Momentum: {len(MOMENTUM_FEATURE_COLUMNS)}')
print(f'  New:      {len(NEW_FEATURES)}')
print(f'  Targets:  max_return_60d, max_drawdown_60d, spy_max_return_60d, alpha_60d')
print(f'  Horizon:  {HORIZON_DAYS} dias')

In [ ]:
# Geração dos alertas sintéticos a partir do price_cache
# Critério: dias em que o ticker caiu >= 3% com RSI < 40
# (replica a lógica do DipRadar para gerar exemplos de treino históricos)

# Mapa ticker → sector (usando sectors.py se disponível)
SECTOR_MAP = {
    'AAPL': 'Technology', 'MSFT': 'Technology', 'GOOGL': 'Technology',
    'AMZN': 'Consumer Cyclical', 'META': 'Technology', 'NVDA': 'Technology',
    'TSLA': 'Consumer Cyclical', 'BRK-B': 'Financial Services',
    'JNJ': 'Healthcare', 'UNH': 'Healthcare', 'LLY': 'Healthcare',
    'ABBV': 'Healthcare', 'MRK': 'Healthcare', 'ABT': 'Healthcare',
    'TMO': 'Healthcare', 'DHR': 'Healthcare',
    'JPM': 'Financial Services', 'BAC': 'Financial Services',
    'WFC': 'Financial Services', 'GS': 'Financial Services',
    'MS': 'Financial Services', 'BLK': 'Financial Services',
    'PG': 'Consumer Defensive', 'KO': 'Consumer Defensive',
    'PEP': 'Consumer Defensive', 'WMT': 'Consumer Defensive',
    'COST': 'Consumer Defensive', 'MCD': 'Consumer Cyclical',
    'HD': 'Consumer Cyclical', 'TGT': 'Consumer Defensive',
    'XOM': 'Energy', 'CVX': 'Energy', 'COP': 'Energy', 'SLB': 'Energy',
    'NEE': 'Utilities', 'DUK': 'Utilities', 'SO': 'Utilities',
    'CAT': 'Industrials', 'DE': 'Industrials', 'HON': 'Industrials',
    'MMM': 'Industrials', 'GE': 'Industrials', 'RTX': 'Industrials',
    'VZ': 'Communication Services', 'T': 'Communication Services',
    'CMCSA': 'Communication Services',
    'AMT': 'Real Estate', 'PLD': 'Real Estate', 'EQIX': 'Real Estate',
    'LIN': 'Basic Materials', 'APD': 'Basic Materials', 'SHW': 'Basic Materials',
    'SPY': 'Unknown', 'QQQ': 'Technology', 'IWM': 'Unknown',
}

# Tenta obter sector do módulo sectors.py
try:
    from sectors import get_sector
    print('sectors.py disponível ✅')
except ImportError:
    get_sector = None
    print('sectors.py não disponível — usando SECTOR_MAP manual')

def get_ticker_sector(ticker: str) -> str:
    if get_sector:
        try:
            s = get_sector(ticker)
            if s:
                return s
        except Exception:
            pass
    return SECTOR_MAP.get(ticker, 'Unknown')

# Gerar alertas históricos sintéticos
print('\nA gerar alertas históricos sintéticos a partir do price_cache...')
alert_records = []

for ticker, df_p in price_cache.items():
    if len(df_p) < 30:
        continue
    sector = get_ticker_sector(ticker)
    closes = df_p['Close']
    returns = closes.pct_change()

    for i in range(20, len(df_p) - HORIZON_DAYS):
        date = df_p.index[i]
        daily_ret = returns.iloc[i]
        if pd.isna(daily_ret) or daily_ret > -0.03:
            continue
        rsi_val = _rsi(closes.iloc[max(0, i-30):i+1])
        if rsi_val > 42:
            continue
        alert_records.append({
            'ticker': ticker,
            'alert_date': date,
            'sector': sector,
        })

base_df = pd.DataFrame(alert_records)
base_df['alert_date'] = pd.to_datetime(base_df['alert_date'])
base_df = base_df.sort_values('alert_date').reset_index(drop=True)

print(f'\n✅ Alertas gerados: {len(base_df)}')
print(f'   Tickers únicos: {base_df["ticker"].nunique()}')
print(f'   Período: {base_df["alert_date"].min().date()} → {base_df["alert_date"].max().date()}')
base_df.head(3)

In [ ]:
# Build do dataset v31 — linha-a-linha
from ml_training.data import compute_sector_alert_count_7d

spy_hist = etf_cache.get(DEFAULT_ETF)
sector_count_lookup = compute_sector_alert_count_7d(base_df)

rows_v31 = []
skipped = {'no_price': 0, 'short_history': 0, 'no_target': 0, 'no_spy_target': 0}

total = len(base_df)
print(f'A construir dataset — {total} alertas...')

for idx, row in base_df.iterrows():
    if (idx + 1) % 500 == 0:
        print(f'  {idx+1}/{total}...')

    ticker = row['ticker']
    alert_date = pd.Timestamp(row['alert_date'])
    sector = row.get('sector', 'Unknown') or 'Unknown'
    etf = SECTOR_ETF.get(sector, DEFAULT_ETF)

    ohlcv = price_cache.get(ticker)
    if ohlcv is None:
        skipped['no_price'] += 1
        continue

    hist = ohlcv[ohlcv.index <= alert_date]
    if len(hist) < 25:
        skipped['short_history'] += 1
        continue

    # Features v1 base (fallback)
    fv = {c: _FALLBACK.get(c, 0.0) for c in FEATURE_COLUMNS}
    add_derived_features(fv)

    # Features v2 price-based
    fv.update(build_v2_features(ohlcv, alert_date))

    # Features v3 momentum
    sec_hist = etf_cache.get(etf)
    sec_slice = sec_hist[sec_hist.index <= alert_date] if sec_hist is not None else None
    spy_slice = spy_hist[spy_hist.index <= alert_date] if spy_hist is not None else None
    add_momentum_features(fv, hist, sec_slice, spy_slice)

    # Features v3.1 NEW
    fv['relative_drop'] = float(fv.get('drop_pct_today', 0.0) - fv.get('sector_drawdown_5d', 0.0))
    fv['sector_alert_count_7d'] = float(sector_count_lookup.get((ticker, alert_date), 0))
    fv['days_since_52w_high'] = days_since_52w_high(ohlcv, alert_date)
    fv['month_of_year'] = float(alert_date.month)

    # Targets
    tgt = build_targets(alert_date, ohlcv)
    if math.isnan(tgt['max_return_60d']):
        skipped['no_target'] += 1
        continue

    spy_max_ret = spy_max_return_forward(spy_hist, alert_date)
    if math.isnan(spy_max_ret):
        skipped['no_spy_target'] += 1
        continue

    alpha_60d = tgt['max_return_60d'] - spy_max_ret

    rec = {
        'ticker': ticker,
        'alert_date': alert_date,
        'sector': sector,
        **{c: fv.get(c, 0.0) for c in FEATURE_COLS_V31},
        'max_return_60d': tgt['max_return_60d'],
        'max_drawdown_60d': tgt['max_drawdown_60d'],
        'spy_max_return_60d': spy_max_ret,
        'alpha_60d': alpha_60d,
    }
    rows_v31.append(rec)

df_v31 = pd.DataFrame(rows_v31)
print(f'\n✅ Dataset construído: {df_v31.shape}')
print(f'   Skipped: {skipped}')
if not df_v31.empty:
    print(f'   Período: {df_v31["alert_date"].min().date()} → {df_v31["alert_date"].max().date()}')
    print(f'   alpha_60d > 5%: {(df_v31["alpha_60d"] > 0.05).mean():.1%}')
df_v31.head(3)

## 📊 Célula 4 — Walk-forward CV + seleção do champion

In [ ]:
import logging
from ml_training.models import get_model_configs
from ml_training.train import run_walk_forward_cv, summarize_results, select_champion
from ml_training.config import N_FOLDS, PURGE_DAYS

logging.basicConfig(level=logging.INFO, format='%(message)s')

assert not df_v31.empty, 'df_v31 está vazio — verifica a Célula 3'

model_configs = get_model_configs(FEATURE_COLS_V31)
print(f'Modelos candidatos: {list(model_configs.keys())}')
print(f'CV: {N_FOLDS} folds, purge={PURGE_DAYS}d')
print(f'Dataset: {len(df_v31)} amostras')
print('\nA correr walk-forward CV...')

results, oof_pred, fold_specs = run_walk_forward_cv(
    df_v31=df_v31,
    model_configs=model_configs,
    n_folds=N_FOLDS,
    purge_days=PURGE_DAYS,
)

summary = summarize_results(results)
print('\n=== Resumo CV ===')
print(summary.to_string(index=False))

In [ ]:
champion_name, champion_row = select_champion(summary)
print(f'\n🏆 Champion: {champion_name}')
print(f'   rho_alpha_mean : {champion_row["rho_alpha_mean"]:.4f}')
print(f'   topk_pnl_mean  : {champion_row["topk_pnl_mean"]:.4f}')
print(f'   n_folds        : {int(champion_row["n_folds"])}')

## 🎓 Célula 5 — Treino full + calibrador isotónico

In [ ]:
import numpy as np
from ml_training.train import train_full_champion, fit_isotonic_calibrator

champion_cfg = model_configs[champion_name]

print('A treinar champion no dataset completo...')
champ_alpha, champ_down, feats_used, n_train = train_full_champion(df_v31, champion_cfg)
print(f'✅ Treinado: {n_train} amostras, {len(feats_used)} features')

# Calibrador isotónico (OOF)
oof_champion = oof_pred[champion_name]
alpha_true = df_v31['alpha_60d'].values

iso_model, brier_oof, n_oof = fit_isotonic_calibrator(
    oof_pred_champion=oof_champion,
    alpha_true=alpha_true,
    alpha_threshold=0.05,
)
print(f'✅ Calibrador OOF: brier={brier_oof:.4f}, n_oof={n_oof}')

# Win-rate alpha
valid_mask = np.isfinite(oof_champion)
win_rate_alpha = float((alpha_true[valid_mask] > 0.05).mean()) if valid_mask.sum() > 0 else 0.0
print(f'   win_rate_alpha (>5%): {win_rate_alpha:.3f}')

## 📦 Célula 6 — Bundle + Report

In [ ]:
from datetime import datetime
from ml_training.bundle import DipModelsV3, save_bundle, build_report, save_report
from ml_training.config import HORIZON_DAYS

champ_hist = results[champion_name]
rho_alpha_vals = [h['rho_alpha'] for h in champ_hist if np.isfinite(h['rho_alpha'])]
rho_down_vals  = [h['rho_down']  for h in champ_hist if np.isfinite(h['rho_down'])]
pnl_vals       = [h['topk_pnl']  for h in champ_hist if np.isfinite(h['topk_pnl'])]

bundle = DipModelsV3(
    model_up         = champ_alpha,
    model_down       = champ_down,
    feature_cols     = feats_used,
    score_calibrator = iso_model,
    n_train_samples  = n_train,
    train_date       = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    champion_name    = champion_name,
    schema_version   = 3,
    momentum_feats   = MOMENTUM_FEATURE_COLUMNS,
    rho_mean         = float(np.mean(rho_alpha_vals)) if rho_alpha_vals else None,
    rho_alpha        = float(np.mean(rho_alpha_vals)) if rho_alpha_vals else None,
    rho_down         = float(np.mean(rho_down_vals))  if rho_down_vals  else None,
    topk_pnl         = float(np.mean(pnl_vals))       if pnl_vals       else None,
    fold_metrics     = champ_hist,
)

PKL_PATH    = Path('/content/dip_models_v3.pkl')
REPORT_PATH = Path('/content/ml_report_v3.json')

save_bundle(bundle, PKL_PATH)
print(f'✅ Bundle: {PKL_PATH} ({PKL_PATH.stat().st_size/1024:.1f} KB)')

report = build_report(
    bundle         = bundle,
    summary_df     = summary,
    brier_oof      = brier_oof,
    win_rate_alpha = win_rate_alpha,
    n_folds_used   = len(champ_hist),
    purge_days     = PURGE_DAYS,
    horizon_days   = HORIZON_DAYS,
    new_features   = NEW_FEATURES,
)
save_report(report, REPORT_PATH)
print(f'✅ Report: {REPORT_PATH}')
print(f"   Champion : {report['champion']}")
print(f"   rho_alpha: {report['metrics']['rho_alpha_mean']}")
print(f"   topk_pnl : {report['metrics']['topk_pnl_mean']}")
print(f"   brier_oof: {report['metrics']['brier_oof']}")

## 💾 Célula 7 — Guardar ml_training_base.parquet

Guarda o dataset construído como `ml_training_base.parquet` no Colab.

In [ ]:
PARQUET_PATH = Path('/content/ml_training_base.parquet')
df_v31.to_parquet(PARQUET_PATH, index=False)
print(f'✅ ml_training_base.parquet: {PARQUET_PATH.stat().st_size/1024:.0f} KB')
print(f'   Linhas  : {len(df_v31)}')
print(f'   Colunas : {len(df_v31.columns)}')

## 📤 Célula 8A — Download direto dos ficheiros

Opção A (manual): download dos 3 ficheiros para o PC.

In [ ]:
from google.colab import files

print('A iniciar downloads...')
files.download(str(PKL_PATH))
files.download(str(REPORT_PATH))
files.download(str(PARQUET_PATH))
print('✅ 3 ficheiros prontos para download:')
print('   📦 dip_models_v3.pkl')
print('   📊 ml_report_v3.json')
print('   🗃️  ml_training_base.parquet')

## 🚀 Célula 8B — Push automático para GitHub (recomendado)

Push direto dos 3 ficheiros para o repo. O Railway deteta e redeploy automaticamente.

In [ ]:
import base64
import json as _json
import requests as _req
from datetime import datetime as _dt

GH_API = 'https://api.github.com'
HEADERS = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github.v3+json',
}

def gh_get_sha(filepath: str):
    url = f'{GH_API}/repos/{REPO_OWNER}/{REPO_NAME}/contents/{filepath}'
    r = _req.get(url, headers=HEADERS, params={'ref': BRANCH})
    return r.json().get('sha') if r.status_code == 200 else None

def gh_push_file(filepath: str, local_path: Path, commit_msg: str):
    content_b64 = base64.b64encode(local_path.read_bytes()).decode()
    sha = gh_get_sha(filepath)
    url = f'{GH_API}/repos/{REPO_OWNER}/{REPO_NAME}/contents/{filepath}'
    payload = {'message': commit_msg, 'content': content_b64, 'branch': BRANCH}
    if sha:
        payload['sha'] = sha
    r = _req.put(url, headers=HEADERS, data=_json.dumps(payload))
    if r.status_code in (200, 201):
        verb = 'Atualizado' if sha else 'Criado'
        print(f'  ✅ {verb}: {filepath}')
    else:
        print(f'  ❌ Erro em {filepath}: {r.status_code} — {r.text[:200]}')

ts = _dt.utcnow().strftime('%Y-%m-%d %H:%M UTC')
rho_str = f'{bundle.rho_alpha:.4f}' if bundle.rho_alpha is not None else 'N/A'
commit_msg = f'chore(ml): retrain [{ts}] champion={champion_name} rho={rho_str}'

print(f'Push para {REPO_OWNER}/{REPO_NAME}@{BRANCH}')
print(f'Commit: {commit_msg}\n')

gh_push_file('dip_models_v3.pkl',        PKL_PATH,    commit_msg)
gh_push_file('ml_report_v3.json',        REPORT_PATH, commit_msg)
gh_push_file('ml_training_base.parquet', PARQUET_PATH, commit_msg)

print('\n🚀 Push concluído — Railway irá redeploy automaticamente.')